Planner → Research → Writer → Reviewer → Executor → Memory

System Architecture
-------------------

                    USER
                      │
                      ▼
             🤖 Planner Agent
      Understands and breaks down the task
                      │
                      ▼
            🤖 Research Agent
      Collects required information
                      │
                      ▼
             🤖 Writer Agent
        Generates Python code
                      │
                      ▼
            🤖 Reviewer Agent
      Reviews and improves the code
                      │
               APPROVED?
             No ─────┐
                     │
                     ▼
              Writer Agent
                     │
             Yes ────┘
                     ▼
            🤖 Executor Agent
        Executes the approved code
                     │
                     ▼
             🤖 Memory Agent
      Saves the task, code, and result

Responsibilities of Each Agent

| Agent       | Role                  | Responsibility                                                                                                       |
| ----------- | --------------------- | -------------------------------------------------------------------------------------------------------------------- |
| 🤖 Planner  | Task Planner          | Breaks the user's request into steps                                                                                 |
| 🔍 Research | Information Collector | Gathers required facts or context (for now, using Gemini; later this can be connected to web search or a RAG system) |
| 💻 Writer   | Code Generator        | Writes Python code based on the plan and research                                                                    |
| 🔎 Reviewer | Quality Checker       | Reviews, improves, and approves the code                                                                             |
| ⚙️ Executor | Code Runner           | Executes the approved code                                                                                           |
| 🧠 Memory   | Conversation Memory   | Stores the task, plan, code, and result for later reuse                                                              |


Multi-agent collaboration,
Task planning,
Research before coding,
Code review workflow,
Code execution,
Memory management,
Passing information between agents.

Project Flow

User:
"Analyze student marks"

        │

Planner:
Creates execution plan

        │

Research:
Explains formulas for Total, Average,
and Percentage

        │

Writer:
Writes Python code

        │

Reviewer:
Reviews and approves

        │

Executor:
Runs the code

        │

Memory:
Stores

• User Task
• Plan
• Research Notes
• Final Code
• Program Output

Research Agent: Instead of coding immediately, the system first gathers relevant information or reasoning that can help solve the task.

Memory Agent: The system keeps a record of what happened (task, plan, research, code, and output), making it easier to build longer-running agent workflows.



Import Libraries

In [1]:
from google.colab import userdata
from google import genai
import json
from datetime import datetime

| Code       | Purpose                                     |
| ---------- | ------------------------------------------- |
| `userdata` | Reads the Gemini API key from Colab Secrets |
| `genai`    | Google GenAI SDK for Gemini models          |
| `json`     | Saves agent memory in JSON format           |
| `datetime` | Records the execution time                  |


Create Gemini Client

In [2]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY_2")

client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini Client Ready")

✅ Gemini Client Ready


Memory Storage

introduce a Memory Agent. Start by creating a memory structure that every agent can access.

In [3]:
memory = {
    "task": "",
    "plan": "",
    "research": "",
    "code": "",
    "review": "",
    "output": "",
    "timestamp": ""
}

print("✅ Memory Initialized")

✅ Memory Initialized


📘 What is the Memory Agent?

User Task
      │
      ▼
Memory["task"]

Planner
      │
      ▼
Memory["plan"]

Research
      │
      ▼
Memory["research"]

Writer
      │
      ▼
Memory["code"]

Reviewer
      │
      ▼
Memory["review"]

Executor
      │
      ▼
Memory["output"]

Planner Agent

In [11]:
def planner_agent(task):

    prompt = f"""
You are a Planner Agent.

Your responsibilities:

1. Understand the user's task.
2. Break it into logical steps.
3. Return ONLY a numbered execution plan.

Task:
{task}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    plan = response.text.strip()

    # Store in memory
    memory["task"] = task
    memory["plan"] = plan

    return plan

What Happens in This Cell?

User Task
     │
     ▼
Planner Agent
     │
     ▼
Execution Plan
     │
     ▼
Memory

Research Agent

In [5]:
def research_agent(task, plan):

    prompt = f"""
You are a Research Agent.

Your responsibilities:

1. Read the user's task.
2. Read the execution plan.
3. Identify important concepts, formulas, or programming techniques.
4. Provide concise research notes that will help the Writer Agent.
5. Do NOT write Python code.

Task:
{task}

Execution Plan:
{plan}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    research = response.text.strip()

    # Save to Memory
    memory["research"] = research

    return research

📘 What Does the Research Agent Do?

Unlike the Writer Agent, the Research Agent does not write code.

Its purpose is to prepare useful information that the Writer Agent can use.

Workflow

User Task
      │
      ▼
Planner Agent
      │
      ▼
Execution Plan
      │
      ▼
🔍 Research Agent
      │
      ▼
Research Notes
      │
      ▼
Memory["research"]

Create the Task

In [7]:
task = """
Student Marks

Tamil = 90
English = 85
Maths = 95
Science = 92
Social = 88

Calculate:

1. Total
2. Average
3. Percentage

Print the results.
"""

Create the Plan

In [12]:
print("=" * 60)
print("PLANNER AGENT")
print("=" * 60)

plan = planner_agent(task)

print(plan)

PLANNER AGENT
1. Store the given marks for each subject: Tamil (90), English (85), Maths (95), Science (92), and Social (88).
2. Calculate the Total marks by summing the scores of all five subjects.
3. Calculate the Average marks by dividing the Total marks by the total number of subjects (5).
4. Calculate the Percentage by dividing the Total marks by the maximum possible total marks (500) and multiplying by 100.
5. Print the calculated Total, Average, and Percentage values.


Test the Research Agent

In [15]:
print("=" * 60)
print("RESEARCH AGENT")
print("=" * 60)

research = research_agent(task, plan)

print(research)

RESEARCH AGENT
### Research Notes for Writer Agent

#### Key Concepts & Formulas

1. **Total Marks:**
   * **Concept:** The cumulative sum of all individual subject scores.
   * **Formula:** $\text{Total} = \text{Tamil} + \text{English} + \text{Maths} + \text{Science} + \text{Social}$
   * **Values:** $90 + 85 + 95 + 92 + 88 = 450$

2. **Average Marks:**
   * **Concept:** The mean mark per subject.
   * **Formula:** $\text{Average} = \frac{\text{Total Marks}}{\text{Total Number of Subjects}}$
   * **Values:** $\frac{450}{5} = 90.0$

3. **Percentage:**
   * **Concept:** The proportion of total marks earned relative to the maximum possible marks, expressed as a fraction of 100.
   * **Assumption:** Each subject is out of 100 marks (Maximum Total = 500).
   * **Formula:** $\text{Percentage} = \left(\frac{\text{Total Marks}}{\text{Maximum Possible Marks}}\right) \times 100$
   * **Values:** $\left(\frac{450}{500}\right) \times 100 = 90.0\%$

---

#### Programming Techniques & Implementatio

Research Agent is working perfectly.

✅ It analyzed the task.

✅ It explained the formulas.

✅ It suggested programming best practices.

✅ It did not write Python code (which is correct).



Writer Agent

Writer Agent uses three inputs:

User Task,
Planner's Plan,
Research Agent's Notes.

In [16]:
def writer_agent(task, plan, research, feedback=""):

    prompt = f"""
You are a Senior Python Developer.

Your responsibilities:

1. Read the user task.
2. Follow the execution plan.
3. Use the research notes.
4. If reviewer feedback exists, improve the code.
5. Return ONLY executable Python code.
6. Do NOT include markdown like ```python.

User Task:
{task}

Execution Plan:
{plan}

Research Notes:
{research}

Reviewer Feedback:
{feedback}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    code = response.text.strip()

    # Remove Markdown if Gemini adds it
    code = code.replace("```python", "")
    code = code.replace("```", "").strip()

    # Save into Memory
    memory["code"] = code

    return code

Test the Writer Agent

In [19]:
print("=" * 60)
print("WRITER AGENT")
print("=" * 60)

code = writer_agent(
    task,
    memory["plan"],
    memory["research"]
)

print(code)

WRITER AGENT
tamil = 90
english = 85
maths = 95
science = 92
social = 88

total = tamil + english + maths + science + social
average = total / 5
percentage = (total / 500) * 100

print(f"Total: {total}")
print(f"Average: {average:.2f}")
print(f"Percentage: {percentage:.2f}%")


Updated Workflow

User
   │
   ▼
Planner Agent
   │
   ▼
Research Agent
   │
   ▼
Writer Agent
   │
   ▼
Memory["code"]

At this stage, the Writer Agent is no longer working from the user request alone—it benefits from the Planner's structured plan and the Research Agent's notes, which is a key characteristic of a collaborative multi-agent system.

Reviewer Agent

Reviews the Writer Agent's code.

Saves the review into memory.

Returns APPROVED or improvement suggestions.

Allows the Writer Agent to use the feedback in the next iteration.

In [20]:
def reviewer_agent(code):

    prompt = f"""
You are a Senior Python Code Reviewer.

Your responsibilities:

1. Review the Python code carefully.
2. Check correctness.
3. Check readability.
4. Check coding best practices.
5. Check variable names.
6. Check formatting.

If everything is correct,
reply ONLY

APPROVED

Otherwise explain what should be improved.

Python Code:

{code}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    review = response.text.strip()

    # Save into Memory
    memory["review"] = review

    return review

Test the Reviewer Agent

In [21]:
print("=" * 60)
print("REVIEWER AGENT")
print("=" * 60)

review = reviewer_agent(memory["code"])

print(review)

REVIEWER AGENT
While the code is functionally correct and produces the expected output, here are a few improvements to align with Python best practices, DRY (Don't Repeat Yourself) principles, and maintainability:

### Improvements

1. **Avoid Magic Numbers**:
   - The numbers `5` and `500` are hardcoded ("magic numbers"). If you add or remove a subject, you would need to update these numbers in multiple places manually.

2. **Use Data Structures for Scalability**:
   - Storing marks in individual variables makes the code rigid. Using a `dict` or a `list` allows you to calculate totals dynamically using `sum()` and count subjects using `len()`.

3. **DRY Principle**:
   - Manually adding variables (`tamil + english + maths + science + social`) becomes error-prone as the list of subjects grows.

---

### Suggested Refactored Code

```python
# Store marks in a dictionary for easy maintenance and expansion
marks = {
    "tamil": 90,
    "english": 85,
    "maths": 95,
    "science": 92,
 

Your Reviewer Agent did not blindly approve the code. Instead, it acted like a Senior Software Engineer and suggested improvements.

That means multi-agent workflow is behaving realistically.

Current Workflow
User
   │
   ▼
🤖 Planner Agent
   │
   ▼
🔍 Research Agent
   │
   ▼
💻 Writer Agent
   │
   ▼
🔎 Reviewer Agent
        │
        ▼
Improvement Suggestions

Now the Reviewer has returned feedback instead of APPROVED

The next step is to send this feedback back to the Writer Agent, allowing it to generate an improved version of the code.

Writer Revision Loop

In [29]:
print("=" * 60)
print("WRITER AGENT - REVISION")
print("=" * 60)

if memory["review"].strip().upper() != "APPROVED":

    improved_code = writer_agent(
        task=memory["task"],
        plan=memory["plan"],
        research=memory["research"],
        feedback=memory["review"]
    )

    memory["code"] = improved_code

    print(improved_code)

else:
    print("Reviewer already approved the code.")

WRITER AGENT - REVISION


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 17.745546294s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

Reviewer has provided feedback, and the Writer has revised the code

Updated Workflow

                 USER
                   │
                   ▼
          🤖 Planner Agent
                   │
                   ▼
         🔍 Research Agent
                   │
                   ▼
          💻 Writer Agent
                   │
                   ▼
         🔎 Reviewer Agent
             │         │
      APPROVED?       NO
         │             │
         │             ▼
         │      💻 Writer Agent
         │      (Uses Feedback)
         │             │
         └─────────────┘
                   │
                   ▼
            APPROVED CODE

This feedback loop is one of the defining characteristics of an agentic AI system: agents collaborate and refine the solution until it meets the review criteria.

Review Again

In [30]:
print("=" * 60)
print("REVIEWER AGENT - FINAL REVIEW")
print("=" * 60)

final_review = reviewer_agent(memory["code"])

memory["review"] = final_review

print(final_review)

REVIEWER AGENT - FINAL REVIEW


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 11.413886193s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

Executor Agent

The Executor Agent is responsible for:

Executing the approved Python code.
Capturing the output.
Saving the output into the shared memory.
Recording the execution timestamp.

In [27]:
from io import StringIO
import sys
from datetime import datetime

def executor_agent(code):

    # Remove Markdown if present
    code = code.replace("```python", "")
    code = code.replace("```", "")
    code = code.strip()

    old_stdout = sys.stdout
    sys.stdout = StringIO()

    try:
        exec(code)

        output = sys.stdout.getvalue()

        memory["output"] = output
        memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        print("Execution Successful")

    except Exception as e:

        output = f"Execution Error:\n{e}"

        memory["output"] = output
        memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    finally:
        sys.stdout = old_stdout

    return output

Execute the Final Code

In [28]:
print("=" * 60)
print("EXECUTOR AGENT")
print("=" * 60)

result = executor_agent(memory["code"])

print(result)

EXECUTOR AGENT
Total: 450
Average: 90.00
Percentage: 90.00%



Memory After Execution

Your memory dictionary now contains:

{
    "task": "...",
    "plan": "...",
    "research": "...",
    "code": "...",
    "review": "...",
    "output": "Total: 450\nAverage: 90.00\nPercentage: 90.00%",
    "timestamp": "2026-07-24 12:35:18"
}

Updated Workflow
                 USER
                   │
                   ▼
        📋 Planner Agent
                   │
                   ▼
        🔍 Research Agent
                   │
                   ▼
        💻 Writer Agent
                   │
                   ▼
        🔎 Reviewer Agent
             │
      Feedback / Approval
             │
             ▼
        💻 Writer Revision
             │
             ▼
        ⚙️ Executor Agent
             │
             ▼
      Program Output Stored

Memory Agent (Workflow Summary)

Memory Agent summarizes everything that happened during the multi-agent workflow.

In [31]:
def memory_agent():

    print("=" * 70)
    print("MULTI-AGENT WORKFLOW SUMMARY")
    print("=" * 70)

    print("\n📋 USER TASK")
    print("-" * 70)
    print(memory["task"])

    print("\n📝 EXECUTION PLAN")
    print("-" * 70)
    print(memory["plan"])

    print("\n🔍 RESEARCH NOTES")
    print("-" * 70)
    print(memory["research"])

    print("\n💻 FINAL PYTHON CODE")
    print("-" * 70)
    print(memory["code"])

    print("\n🔎 REVIEW RESULT")
    print("-" * 70)
    print(memory["review"])

    print("\n⚙️ EXECUTION OUTPUT")
    print("-" * 70)
    print(memory["output"])

    print("\n🕒 EXECUTION TIME")
    print("-" * 70)
    print(memory["timestamp"])

    print("\n" + "=" * 70)
    print("PROJECT 4 COMPLETED SUCCESSFULLY")
    print("=" * 70)

Run the Memory Agent

In [32]:
print("=" * 70)
print("MEMORY AGENT")
print("=" * 70)

memory_agent()

MEMORY AGENT
MULTI-AGENT WORKFLOW SUMMARY

📋 USER TASK
----------------------------------------------------------------------

Student Marks

Tamil = 90
English = 85
Maths = 95
Science = 92
Social = 88

Calculate:

1. Total
2. Average
3. Percentage

Print the results.


📝 EXECUTION PLAN
----------------------------------------------------------------------
1. Store the given marks for each subject: Tamil (90), English (85), Maths (95), Science (92), and Social (88).
2. Calculate the Total marks by summing the scores of all five subjects.
3. Calculate the Average marks by dividing the Total marks by the total number of subjects (5).
4. Calculate the Percentage by dividing the Total marks by the maximum possible total marks (500) and multiplying by 100.
5. Print the calculated Total, Average, and Percentage values.

🔍 RESEARCH NOTES
----------------------------------------------------------------------
### Research Notes for Writer Agent

#### Key Concepts & Formulas

1. **Total Marks:**

Multi-Agent AI System:
----------------------

| Agent             | Responsibility                     | Status |
| ----------------- | ---------------------------------- | ------ |
| 📋 Planner Agent  | Creates execution plan             | ✅      |
| 🔍 Research Agent | Provides research notes            | ✅      |
| 💻 Writer Agent   | Generates Python code              | ✅      |
| 🔎 Reviewer Agent | Reviews and suggests improvements  | ✅      |
| ⚙️ Executor Agent | Executes the generated code        | ✅      |
| 🧠 Memory Agent   | Stores and summarizes the workflow | ✅      |


                 USER
                   │
                   ▼
          📋 Planner Agent
                   │
                   ▼
         🔍 Research Agent
                   │
                   ▼
          💻 Writer Agent
                   │
                   ▼
         🔎 Reviewer Agent
             │         │
     APPROVED?         NO
         │             │
         │             ▼
         │      💻 Writer Revision
         │             │
         └─────────────┘
                   │
                   ▼
          ⚙️ Executor Agent
                   │
                   ▼
           🧠 Memory Agent

Overall Evaluation
------------------


| Feature            | Rating |
| ------------------ | :----: |
| Multi-Agent Design |  ⭐⭐⭐⭐⭐ |
| Shared Memory      |  ⭐⭐⭐⭐⭐ |
| Planner            |  ⭐⭐⭐⭐⭐ |
| Research           |  ⭐⭐⭐⭐⭐ |
| Writer             |  ⭐⭐⭐⭐⭐ |
| Reviewer           |  ⭐⭐⭐⭐⭐ |
| Executor           |  ⭐⭐⭐⭐⭐ |
| Memory Summary     |  ⭐⭐⭐⭐⭐ |
| Code Quality       |  ⭐⭐⭐⭐⭐ |


Recommended Next Projects

To continue building more advanced systems, here's a natural progression:

| Project          | New Capability                                                     |
| ---------------- | ------------------------------------------------------------------ |
| ✅ Project 1      | Writer + Reviewer                                                  |
| ✅ Project 2      | Two-Agent Conversation                                             |
| ✅ Project 3      | Four-Agent System                                                  |
| ✅ Project 4      | Memory + Research + Execution                                      |
| 🚀 **Project 5** | **Tool-Using AI Agents (Calculator, File Reader, Web Search)**     |
| 🚀 **Project 6** | **RAG Multi-Agent System (FAISS + PDFs)**                          |
| 🚀 **Project 7** | **Autonomous AI Team (Manager, Planner, Coder, Tester, Debugger)** |
| 🚀 **Project 8** | **Business AI Assistant with Real APIs and Databases**             |


====

Given your progression, Project 5: Tool-Using AI Agents is the next logical step. It will introduce agents that can invoke external tools (such as calculators, file readers, or web search) instead of relying only on language-model reasoning, making the system significantly more capable.